In [18]:
import os
# Force the PyTorch backend for Keras 3
os.environ["KERAS_BACKEND"] = "torch"
current_dir = os.getcwd()
import numpy as np
import keras
from keras import layers
import torch
import torch.nn as nn
# =====================================================================
# Loading and saving images
# =====================================================================
from PIL import Image
import torchvision.transforms as T
from pathlib import Path
# =====================================================================
# Downloading the model
# =====================================================================
import urllib.request

In [2]:
# =====================================================================
# Custom normalization function
# =====================================================================
class InstanceNormalization(keras.layers.Layer):
    """
    Framework-agnostic Instance Normalization layer for Keras 3 (channels_last format).
    Calculates mean and variance across spatial dimensions [Height, Width] dynamically.
    """
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        # input_shape is (Batch, Height, Width, Channels)
        channel_axis = -1
        num_channels = input_shape[channel_axis]
        
        # Learnable scale (gamma) and shift (beta) parameters per channel
        self.gamma = self.add_weight(
            name="gamma",
            shape=(num_channels,),
            initializer="ones",
            trainable=True
        )
        self.beta = self.add_weight(
            name="beta",
            shape=(num_channels,),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        # Calculate mean and variance across Height (axis 1) and Width (axis 2)
        # keepdims=True leaves the shape as (Batch, 1, 1, Channels) so it broadcasts nicely
        mean = keras.ops.mean(inputs, axis=[1, 2], keepdims=True)
        variance = keras.ops.var(inputs, axis=[1, 2], keepdims=True)
        
        # Standardize the input features
        normalized = (inputs - mean) / keras.ops.sqrt(variance + self.epsilon)
        
        # Apply learnable scale and shift parameters
        return self.gamma * normalized + self.beta



In [10]:
# =====================================================================
# Functions for setting up the generator
# =====================================================================
def downsample_win(filters, size, apply_norm=True):
    initializer = keras.initializers.RandomNormal(0., 0.02)
    result = keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding='same', 
                             kernel_initializer=initializer, use_bias=False))
    if apply_norm:
        result.add(InstanceNormalization())  # <-- Swapped here
    result.add(layers.LeakyReLU(0.2))
    return result

def downsample_win_s(filters, size, apply_norm=True):
    initializer = keras.initializers.RandomNormal(0., 0.02)
    result = keras.Sequential()
    result.add(layers.SeparableConv2D(filters, size, strides=2, padding='same', 
                                      depthwise_initializer=initializer, 
                                      pointwise_initializer=initializer, use_bias=False))
    if apply_norm:
        result.add(InstanceNormalization())  # <-- Swapped here
    result.add(layers.LeakyReLU(0.2))
    return result

def upsample_win(filters, size, apply_dropout=False):
    initializer = keras.initializers.RandomNormal(0., 0.02)
    result = keras.Sequential()
    result.add(layers.UpSampling2D(size=2, interpolation='nearest'))
    result.add(layers.Conv2D(filters, size, strides=1, padding='same', 
                             kernel_initializer=initializer, use_bias=False))
    result.add(InstanceNormalization())      # <-- Swapped here
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def Generator():
    inputs = layers.Input(shape=[256, 256, 3])

    down_stack = [
        downsample_win(64, 4, apply_norm=False), 
        downsample_win(128, 4),                  
        downsample_win(256, 4),                  
        downsample_win(512, 4),                  
        downsample_win(512, 4),                  
        downsample_win(512, 4),                  
        downsample_win(512, 4),                  
    ]
    
    down_stack_s = [
        downsample_win_s(64, 4, apply_norm=False),
        downsample_win_s(128, 4),
        downsample_win_s(256, 4),
        downsample_win_s(512, 4),
        downsample_win_s(512, 4),
        downsample_win_s(512, 4),
        downsample_win_s(512, 4),
    ]

    up_stack = [
        upsample_win(512, 4, apply_dropout=True), 
        upsample_win(512, 4, apply_dropout=True), 
        upsample_win(512, 4, apply_dropout=True), 
        upsample_win(256, 4),                     
        upsample_win(128, 4),                     
        upsample_win(64, 4),                      
    ]

    # Encoder 1 Pass
    x_enc = inputs
    skips = []
    for down in down_stack:
        x_enc = down(x_enc)
        skips.append(x_enc)
    
    # Encoder 2 Pass
    x_sep = inputs
    skips_s = []
    for down in down_stack_s:
        x_sep = down(x_sep)
        skips_s.append(x_sep)

    # Bottleneck Isolation
    bottleneck_concat = layers.Concatenate()([x_enc, x_sep])
    current_decoder_tensor = layers.Conv2D(512, 1, padding='same', use_bias=False)(bottleneck_concat)

    skips = list(reversed(skips[:-1]))
    skips_s = list(reversed(skips_s[:-1]))

    # FIX: Use index-specific, immutable tensor tracking variables
    # to prevent PyTorch memory overwrite clashes
    for i, up in enumerate(up_stack):
        up_scaled = up(current_decoder_tensor)
        merged = layers.Concatenate()([up_scaled, skips[i], skips_s[i]])
        
        out_filters = up_stack[i+1].layers[1].filters if i < len(up_stack)-1 else 64
        current_decoder_tensor = layers.Conv2D(out_filters, 1, padding='same', use_bias=False)(merged)

    # Final Output Rendering
    initializer = keras.initializers.RandomNormal(0., 0.02)
    final_upsample = layers.UpSampling2D(size=2, interpolation='nearest')(current_decoder_tensor)
    outputs = layers.Conv2D(3, 4, strides=1, padding='same', 
                            kernel_initializer=initializer, activation='tanh')(final_upsample)

    return keras.Model(inputs=inputs, outputs=outputs, name="Modern_WNet_Generator")

# =====================================================================
# Image processing functions
# =====================================================================


def create_blend_mask(patch_size=256):
    """Creates a 2D weight mask that tapers to 0 at the edges to blend overlapping regions."""
    # Linear ramp from 0 to 1, then back to 0
    ramp = np.linspace(0, 1, patch_size // 2)
    ramp = np.concatenate([ramp, ramp[::-1]])
    
    # 2D mask via outer product
    mask_2d = np.outer(ramp, ramp)
    
    # Keep a small flat zone in the dead center if desired, or just use the ramp
    # To avoid dividing by absolute zero later, clip the minimum weight slightly
    mask_2d = np.clip(mask_2d, 1e-4, 1.0)
    return torch.from_numpy(mask_2d).float()

def process_large_image(image, model, patch_size=256, stride=128, device="cuda"):
    """
    Chops, normalizes to [-1, 1], processes, denormalizes back to [0, 1], 
    and seamlessly fuses a large image.
    """
    model.eval()
    _, H, W = image.shape
    
    # Canvases to accumulate predictions and weights (on [0, 1] scale)
    canvas = torch.zeros((3, H, W), device=device)
    weight_grid = torch.zeros((1, H, W), device=device)
    
    # Create the 2D blending mask, add channel dim to make it (1, 256, 256)
    blend_mask = create_blend_mask(patch_size).to(device).unsqueeze(0)
    
    # Generate coordinates for sliding window
    y_coords = list(range(0, H - patch_size + 1, stride))
    if y_coords[-1] != H - patch_size:
        y_coords.append(H - patch_size)
        
    x_coords = list(range(0, W - patch_size + 1, stride))
    if x_coords[-1] != W - patch_size:
        x_coords.append(W - patch_size)
        
    with torch.no_grad():
        for y in y_coords:
            for x in x_coords:
                # 1. Crop standard patch -> (3, 256, 256) [Values: 0 to 1]
                patch = image[:, y:y+patch_size, x:x+patch_size].to(device)
                
                # == CONVERSION TO [-1, 1] ==
                # Shift 0->1 down to -1->1
                patch = (patch * 2.0) - 1.0
                
                # 2. Add batch dimension -> (1, 3, 256, 256)
                patch_batch = patch.unsqueeze(0)
                
                # 3. Permute to Channels-Last -> (1, 256, 256, 3)
                patch_batch_channels_last = patch_batch.permute(0, 2, 3, 1)
                
                # 4. Feed to model -> Output shape: (1, 256, 256, 3) [Values: -1 to 1]
                prediction_channels_last = model(patch_batch_channels_last) 
                
                # 5. Permute back to PyTorch standard -> (1, 3, 256, 256)
                prediction = prediction_channels_last.permute(0, 3, 1, 2)
                
                # 6. Drop batch dimension -> (3, 256, 256)
                prediction = prediction.squeeze(0)
                
                # == CONVERSION BACK TO [0, 1] ==
                # Reverse the math so blending works smoothly on standard scales
                prediction = (prediction + 1.0) / 2.0
                
                # 7. Apply the blending mask and accumulate
                weighted_prediction = prediction * blend_mask
                canvas[:, y:y+patch_size, x:x+patch_size] += weighted_prediction
                weight_grid[:, y:y+patch_size, x:x+patch_size] += blend_mask
                
    # 8. Final normalization to resolve overlaps perfectly
    final_reconstruction = canvas / weight_grid
    
    return final_reconstruction.cpu()

# =====================================================================
# Loading and saving images
# =====================================================================
def load_image_to_tensor(image_path):
    """Opens an image (JPG, PNG, or TIFF) and converts it to a PyTorch float tensor."""
    ext = os.path.splitext(image_path)[-1].lower()

    # Default fallback for JPG, PNG, and basic TIFFs using Pillow
    img = Image.open(image_path).convert('RGB')
    
    # Converts a PIL Image (H, W, C) to a Tensor (C, H, W) scaled between 0.0 and 1.0
    transform = T.ToTensor()
    return transform(img)


def save_tensor_to_image(tensor, output_path):
    """Converts a PyTorch tensor (C, H, W) back into an image file."""
    # Clamp to safely handle any out-of-bounds pixels from the GAN
    tensor = torch.clamp(tensor, 0.0, 1.0)
    
    # Transform back to a PIL Image
    transform = T.ToPILImage()
    img = transform(tensor)
    
    # Save based on file extension
    img.save(output_path)
    print(f"Successfully saved fused prediction to: {output_path}")

def change_extension(filename: str, new_ext: str) -> str:
    """
    Changes the extension of a filename string to the specified new extension.
    
    :param filename: The original file name or path (e.g., "image.png" or "/path/to/data.tif")
    :param new_ext: The target extension (e.g., "jpg", ".jpg", or "npy")
    :return: The updated filename string with the new extension.
    """
    # 1. Clean up the input extension (ensure it starts with a dot '.')
    if not new_ext.startswith('.'):
        new_ext = f".{new_ext}"
        
    # 2. Use pathlib to swap the extension and return it as a string
    return str(Path(filename).with_suffix(new_ext))

In [7]:
# =====================================================================
# LOAD W-SEG GENERATOR
# =====================================================================
generator = Generator()
print("Loading previous weights...")
generator.load_weights(current_dir + '/model_checkpoint/Tgen_weights.weights.h5')

Loading previous weights...


In [17]:

# =====================================================================
# EXAMPLE USAGE PIPELINE
# =====================================================================
img_dir = current_dir+'/OneDrive_1_7-1-2026/'
img_fname = '260629_TBO_test_63x_gly_EDF_Merged_ch00_SV_T.tif'

# Change format if necessary
save_as = 'png'
prefix = 'pred_'
output_file = change_extension(prefix+img_fname, save_as)


if __name__ == "__main__":
    # Setup your paths and device
    input_file = img_dir + img_fname  # Works for .png or .tiff too
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load the image
    print(f"Loading {input_file}...")
    large_image_tensor = load_image_to_tensor(input_file)
    print(f"Loaded image shape: {large_image_tensor.shape}") # Expects (3, H, W)
    
    # Run the sliding window
    print("Processing image through overlapping sliding windows...")
    processed_tensor = process_large_image(
        image=large_image_tensor,
        model=generator,
        patch_size=256,
        stride=128,  # 50% overlap for clean blending
        device=device
    )
    
    # Save the final master image
    save_tensor_to_image(processed_tensor, output_file)

Loading /home/leo/Documents/Python/w-seg/OneDrive_1_7-1-2026/260629_TBO_test_63x_gly_EDF_Merged_ch00_SV_T.tif...
Loaded image shape: torch.Size([3, 4042, 10550])
Processing image through overlapping sliding windows...
Successfully saved fused prediction to: pred_260629_TBO_test_63x_gly_EDF_Merged_ch00_SV_T.png
